In [ ]:
# this file will test the XFuse Module on some example porespy data


In [ ]:
import os
torch_dir = '/dls/science/groups/imaging/infuse/XRDCT_Fusion/torch_cache/'
os.environ["HUGGINGFACE_HUB_CACHE"] = torch_dir
os.environ["TORCH_HOME"] = torch_dir
os.environ["HF_HUB_CACHE"] = torch_dir
os.environ["HF_HOME"] = torch_dir

In [ ]:
from x_fuse.utils import get_multiphase, downsample_xrdct, xrd_to_tensor
from x_fuse.fusion import XFuse
import numpy as np
import porespy as ps
import matplotlib.pyplot as plt
import torch
import porespy as ps
from PIL import Image, ImageOps
import hr_dv2.transform as tr
from hr_dv2.utils import do_single_pca, rescale_pca
import torchvision.transforms as transforms
from hr_dv2.high_res import HighResDV2


In [ ]:
# create the dataset

IMG_SIZE = 224
N_SAMPLES = 10

ps_dict = {"blobs": ps.generators.blobs, "rand_spheres": ps.generators.random_spheres}

im1 = ps_dict["blobs"]
im2 = ps_dict["rand_spheres"]

im1_kwargs = {'shape': [IMG_SIZE, IMG_SIZE], 'blobiness': 0.5, 'porosity': 0.1}
im2_kwargs = {'r': 40, 'clearance': 10}

gray_imgs, phase_A_masks, phase_B_masks = get_multiphase(im1, im1_kwargs, im2, im2_kwargs, extract_spheres=True, dataset_size=10)

In [ ]:
gray_imgs, low_A, low_B = downsample_xrdct(gray_imgs, phase_A_masks, phase_B_masks, factor=10)

In [ ]:
# now I have my dataset I can test the XFuse Module

DINO_MODEL = "nope_dinov3_SMALL_1625" #DINOv3
MODEL_PATH = "/dls/science/groups/imaging/infuse/XRDCT_Fusion/dinosaw-env/test/20260413_1615_dinov2_GIANT/best_model.pth" #DINOv2
MODEL_PATH = "/dls/science/groups/imaging/infuse/XRDCT_Fusion/dinosaw-env/models/dinov3_SMALL_1625_nope/best_model.pth" #DINOv3
# MODEL_PATH = "/dls/science/groups/imaging/infuse/DINOv3/Models/alibi_dv2_vits14_reg.pth"

CHK_PATH = "/dls/science/groups/imaging/infuse/DINOv3/Models/"
LIB_PATH = "/dls/science/groups/imaging/infuse/DINOv3/dinov3/"

# PATCH_SIZE = 14 if "dinov2" in DINO_MODEL else 16
PATCH_SIZE = 16
STRIDE = 4
PATCH_H, PATCH_W = IMG_SIZE // PATCH_SIZE, IMG_SIZE // PATCH_SIZE
FUSION_METHOD = 'gating'
LOSS_FN = 'bce'
TOP_K = 10
INVERT = False


if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
# refine the data

idx = -1

xct_img = gray_imgs[idx]
xrd_A = low_A[idx]
xrd_B = low_B[idx]

plt.imshow(xct_img, cmap='gray')
plt.title('XCT Image')
plt.axis('off')
plt.show()

def invert_image(image):
    return np.ones_like(image) - image

if INVERT:
    xct_img = invert_image(xct_img)
    xrd_A = invert_image(xrd_A)
    xrd_B = invert_image(xrd_B)

plt.imshow(xct_img, cmap='gray')
plt.title('XCT Image')
plt.axis('off')
plt.show()

# print(xct_img[0][0])

# xct_img = phase_B_masks[-1]

In [ ]:
# net = XFuse(DINO_MODEL, FUSION_METHOD, stride=STRIDE, pca_dim=-1, track_grad=False, dtype=torch.float16, device=DEVICE, loss_fn=LOSS_FN, model_path = MODEL_PATH)

In [ ]:
h, w = xct_img.shape
print(f"XCT image shape: {xct_img.shape}, Phase A mask shape: {xrd_A.shape}, Phase B mask shape: {xrd_B.shape}")

In [ ]:
xct_img_tr = tr.closest_crop(xct_img, PATCH_H, PATCH_W)
xct_img_tr = tr.get_input_transform(IMG_SIZE, IMG_SIZE)


def load_img(img, transform):
    unnormalize = transforms.Normalize(
        mean=(-0.485/0.229, -0.456/0.224, -0.406/0.225),
        std=(1/0.229, 1/0.224, 1/0.225),
    )
    img_uint8 = (img * 255).astype(np.uint8)
    image = ImageOps.autocontrast(Image.fromarray(img_uint8).convert("RGB"))
    tensor = transform(image)
    trans_img = transforms.ToPILImage()(unnormalize(tensor))
    return tensor, trans_img

In [ ]:
xct_tensor, xct_image = load_img(xct_img, xct_img_tr)
print(xct_tensor.shape)

xct_tensor = xct_tensor.to(torch.float16).to(DEVICE)
print(DEVICE)

xrd_A_tensor = xrd_to_tensor(xrd_A, DEVICE)
xrd_B_tensor = xrd_to_tensor(xrd_B, DEVICE)

print(xct_tensor.device)

In [ ]:
shift_dists = [i for i in range(1,3)]

fwd_shift, inv_shift = tr.get_shift_transforms(shift_dists, 'Moore')
fwd_flip, inv_flip = tr.get_flip_transforms()
fwd, inv = tr.combine_transforms(fwd_shift, fwd_flip, inv_shift, inv_flip)

# net.set_xrd_transforms(xrd_A_tensor, fwd, inv)

print(f"XRD Tensor shape: {xrd_A_tensor.shape}", f"XCT Tensor shape: {xct_img.shape}")


In [ ]:
net_lgA = XFuse(DINO_MODEL, FUSION_METHOD, stride=STRIDE, pca_dim=-1,
               track_grad=False, dtype=torch.float16, device=DEVICE, loss_fn=LOSS_FN, model_path = MODEL_PATH)
# net_lgA.learned_gating.select_top_k(TOP_K)
net_lgA.set_xrd_transforms(xrd_A_tensor, fwd, inv)

# N_EPOCHS = 100
# losses = []

# for epoch in range(N_EPOCHS):
#     loss = net_lgA.train_fusion_step(xct_tensor)
#     losses.append(loss)
#     if epoch % 10 == 0:
#         print(f"Epoch {epoch:>3d}  loss: {loss:.6f}")

# plt.plot(losses)
# plt.xlabel("Epoch")
# plt.ylabel(f"{LOSS_FN} Loss")
# plt.title("Gating MLP training")
# plt.show()

feats_lgA = net_lgA.forward_sequential(xct_tensor, top_k=TOP_K)
feats_np_lgA = tr.to_numpy(feats_lgA[0])
feats_flat_lgA = tr.flatten(feats_np_lgA, feats_np_lgA.shape[1], feats_np_lgA.shape[2], feats_np_lgA.shape[0])
pcaed_lgA = do_single_pca(feats_flat_lgA, 3, n_samples=5000)
rescaled_lgA = rescale_pca(pcaed_lgA)

In [ ]:
net_lgB = XFuse(DINO_MODEL, FUSION_METHOD, stride=STRIDE, pca_dim=-1,
                track_grad=False, dtype=torch.float16, device=DEVICE, loss_fn=LOSS_FN, model_path = MODEL_PATH)
# net_lgB.learned_gating.select_top_k(TOP_K)
net_lgB.set_xrd_transforms(xrd_B_tensor, fwd, inv)

# N_EPOCHS = 100
# losses = []

# for epoch in range(N_EPOCHS):
#     loss = net_lgB.train_fusion_step(xct_tensor)
#     losses.append(loss)
#     if epoch % 10 == 0:
#         print(f"Epoch {epoch:>3d}  loss: {loss:.6f}")

# plt.plot(losses)
# plt.xlabel("Epoch")
# plt.ylabel(f"{LOSS_FN} Loss")
# plt.title("Gating MLP training")
# plt.show()

feats_lgB = net_lgB.forward_sequential(xct_tensor, top_k=TOP_K)

feats_np_lgB = tr.to_numpy(feats_lgB[0])
feats_flat_lgB = tr.flatten(feats_np_lgB, feats_np_lgB.shape[1], feats_np_lgB.shape[2], feats_np_lgB.shape[0])
pcaed_lgB = do_single_pca(feats_flat_lgB, 3, n_samples=5000)
rescaled_lgB = rescale_pca(pcaed_lgB)

In [ ]:
# net2 = HighResDV2(DINO_MODEL, 4, track_grad=False, dtype=torch.float16, device=DEVICE)
# net2.set_transforms(fwd, inv)

# feats_tensor2 = net2.forward_sequential(xct_tensor)

# feats2 = feats_tensor2[0]
# feats_np2 = tr.to_numpy(feats2)
# feats_flat2 = tr.flatten(feats_np2, feats_np2.shape[1], feats_np2.shape[2], feats_np2.shape[0])
# print(min(feats_flat2.flatten()), max(feats_flat2.flatten()))
# pcaed2 = do_single_pca(feats_flat2, 3, n_samples=5000)
# rescaled2= rescale_pca(pcaed2)

In [ ]:
# I want to compare 10 PCA components of feats_flat and feats_flat2 and plot them on top of each other

pcaed_lgA = do_single_pca(feats_flat_lgA, 10, n_samples=5000)
pcaed_lgB = do_single_pca(feats_flat_lgB, 10, n_samples=5000)
# pcaed_2 = do_single_pca(feats_flat2, 10, n_samples=5000)

fig, axs = plt.subplots(nrows=2, ncols=10, figsize=(30, 9))

for i in range(10):
    axs[0, i].imshow(pcaed_lgA[:, i].reshape(IMG_SIZE, IMG_SIZE), cmap='viridis')
    axs[1, i].imshow(pcaed_lgB[:, i].reshape(IMG_SIZE, IMG_SIZE), cmap='viridis')
    # axs[2, i].imshow(pcaed_2[:, i].reshape(IMG_SIZE, IMG_SIZE), cmap='viridis')
    axs[0, i].set_title(f'XFuse LG A PCA {i+1}', fontsize=10)
    axs[1, i].set_title(f'XFuse LG B PCA {i+1}', fontsize=10)
    # axs[2, i].set_title(f'HR-Dv2 PCA {i+1}', fontsize=10)
    axs[0, i].set_axis_off()
    axs[1, i].set_axis_off()
    # axs[2, i].set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.99

mapA = pcaed_lgA[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold
mapB = pcaed_lgB[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold

fig, ax = plt.subplots(2,3, figsize=(18,6))
ax[0,0].imshow(xct_img, cmap='gray');        ax[0,0].set_title('XCT')
ax[0,1].imshow(mapA, cmap='viridis');       ax[1,0].set_title('XFuse LG A PCA 1')
ax[0,2].imshow(mapB, cmap='viridis');       ax[1,1].set_title('XFuse LG B PCA 1')


ax[1,0].imshow(xct_img, cmap='gray')
ax[1,1].imshow(xrd_A, cmap='gray', interpolation='lanczos')
ax[1,2].imshow(xrd_B, cmap='gray', interpolation='lanczos')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(24, 6))
axs[0].imshow(xct_img, cmap='gray');        axs[0].set_title('XCT')
im1 = axs[1].imshow(rescaled_lgA.reshape(IMG_SIZE, IMG_SIZE, 3)); axs[1].set_title('LG A')
im2 = axs[2].imshow(rescaled_lgB.reshape(IMG_SIZE, IMG_SIZE, 3)); axs[2].set_title('LG B')
# im3 = axs[3].imshow(rescaled2.reshape(IMG_SIZE, IMG_SIZE, 3));    axs[3].set_title('HR-Dv2')
for ax in axs: ax.set_axis_off()

# Grayscale intensity colorbars per panel
for ax, im in zip(axs[1:], [im1, im2]):
    sm = plt.cm.ScalarMappable(cmap='gray', norm=plt.Normalize(vmin=0, vmax=1))
    plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()

In [ ]:
THRESHOLD = 0.99

mask_lgA = rescaled_lgA.reshape(IMG_SIZE, IMG_SIZE, 3) > THRESHOLD
mask_lgB = rescaled_lgB.reshape(IMG_SIZE, IMG_SIZE, 3) > THRESHOLD

fig, axs = plt.subplots(1, 5, figsize=(30, 6))
axs[0].imshow(xct_img, cmap='gray');                                    axs[0].set_title('XCT')
axs[1].imshow(rescaled_lgA.reshape(IMG_SIZE, IMG_SIZE, 3), cmap='gray');       axs[1].set_title('LG A')
axs[2].imshow(rescaled_lgB.reshape(IMG_SIZE, IMG_SIZE, 3), cmap='gray');       axs[2].set_title('LG B')
axs[3].imshow(mapA, cmap='gray');  axs[3].set_title(f'LG A thresh ({THRESHOLD})')
axs[4].imshow(mapB, cmap='gray');  axs[4].set_title(f'LG B thresh ({THRESHOLD})')
for ax in axs: ax.set_axis_off()
plt.tight_layout()

In [ ]:
THRESHOLD = 0.5

pca_lgA = rescaled_lgA.reshape(IMG_SIZE, IMG_SIZE, 3)
pca_lgB = rescaled_lgB.reshape(IMG_SIZE, IMG_SIZE, 3)

masks_lgA = [pca_lgA[:, :, c] > THRESHOLD for c in range(3)]
masks_lgB = [pca_lgB[:, :, c] > THRESHOLD for c in range(3)]

fig, axs = plt.subplots(2, 5, figsize=(30, 12))

axs[0, 0].imshow(xct_img, cmap='gray');  axs[0, 0].set_title('XCT')
axs[0, 1].imshow(pca_lgA);              axs[0, 1].set_title('LG A PCA')
axs[1, 0].imshow(xct_img, cmap='gray');  axs[1, 0].set_title('XCT')
axs[1, 1].imshow(pca_lgB);              axs[1, 1].set_title('LG B PCA')

for c in range(3):
    axs[0, c + 2].imshow(masks_lgA[c], cmap='gray')
    axs[0, c + 2].set_title(f'LG A Ch{c} (t={THRESHOLD})')
    axs[1, c + 2].imshow(masks_lgB[c], cmap='gray')
    axs[1, c + 2].set_title(f'LG B Ch{c} (t={THRESHOLD})')

for ax in axs.flat: ax.set_axis_off()
plt.tight_layout()

In [ ]:
import torch.nn.functional as F
xrd_up = F.interpolate(xrd_B_tensor, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
xrd_mask = (xrd_up.squeeze() > xrd_up.mean()).cpu().numpy()  # high-XRD pixels

# Compare mean feature vectors inside vs outside the XRD spots
feats_np = feats_lgB[0].float().cpu().numpy()  # (C, H, W)
inside  = feats_np[:, xrd_mask].mean(axis=1)
outside = feats_np[:, ~xrd_mask].mean(axis=1)

plt.plot(inside - outside)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel('Channel'); plt.ylabel('Mean inside - outside XRD spot')
plt.title('Channel discrimination at XRD locations')

In [ ]:
# Get masks at feature resolution
xrd_A_up = F.interpolate(xrd_A_tensor, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze().cpu().numpy()
xrd_B_up = F.interpolate(xrd_B_tensor, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).squeeze().cpu().numpy()
mask_A = xrd_A_up > 0.5
mask_B = xrd_B_up > 0.5
feats_np = feats_lgA[0].float().cpu().numpy()  # (C, H, W)

mean_A = feats_np[:, mask_A].mean(axis=1)
mean_B = feats_np[:, mask_B].mean(axis=1)
mean_bg = feats_np[:, ~(mask_A | mask_B)].mean(axis=1)

plt.plot(mean_A - mean_bg, label='Phase A vs background')
plt.plot(mean_B - mean_bg, label='Phase B vs background')
plt.plot(mean_A - mean_B, label='Phase A vs Phase B')
plt.axhline(0, color='k', linestyle='--')
plt.legend(); plt.xlabel('Channel'); plt.show()

In [ ]:
import torchvision.transforms as T
unnorm = T.Normalize(
    mean=(-0.485/0.229, -0.456/0.224, -0.406/0.225),
    std=(1/0.229, 1/0.224, 1/0.225)
)
xct_vis = unnorm(xct_tensor.float().cpu()).permute(1,2,0).numpy()
xct_vis = (xct_vis - xct_vis.min()) / (xct_vis.max() - xct_vis.min())

fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(xct_img, cmap='gray');      axs[0].set_title('Raw XCT')
axs[1].imshow(xct_vis);                   axs[1].set_title('After normalisation (fed to DINO)')
axs[2].imshow(xrd_A_up, cmap='hot');      axs[2].set_title('Phase A mask')
for ax in axs: ax.set_axis_off()
plt.tight_layout()

# Refining the fused masks using SAM2

In [ ]:
# define the segmentation masks here using a simple threshold

threshold = 0.5
mapA = pcaed_lgA[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold
mapB = pcaed_lgB[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(18,6))
ax[0,0].imshow(xct_img, cmap='gray');        ax[0,0].set_title('XCT')
ax[0,1].imshow(mapA, cmap='gray');       ax[1,0].set_title('XFuse mask (coarse)')
ax[0,2].imshow(mapB, cmap='gray');       ax[1,1].set_title('XFuse mask (coarse)')
ax[1,0].imshow(xct_img, cmap='gray')
ax[1,1].imshow(xrd_A, cmap='gray', interpolation='lanczos')
ax[1,2].imshow(xrd_B, cmap='gray', interpolation='lanczos')
plt.tight_layout()
plt.show()

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

SAM2_FOLDER = "//dls/science/groups/imaging/infuse/SAM2/" # ensure double //
SAM2_MODEL = "sam2.1_hiera_large"

sam2 = build_sam2(f"{SAM2_FOLDER}{SAM2_MODEL}.yaml", f"{SAM2_FOLDER}{SAM2_MODEL}.pt", device=DEVICE)
predictor = SAM2ImagePredictor(sam2)

In [ ]:
from PIL import ImageOps, Image

xct_pil = Image.fromarray((xct_img * 255).astype(np.uint8)).convert("RGB")
xct_pil = ImageOps.autocontrast(xct_pil)
predictor.set_image(np.array(xct_pil))

In [ ]:
import cv2

# Resize coarse mask to SAM2's expected mask resolution
mask_input = cv2.resize(mapA.astype(np.float32), (256, 256))[None]  # (1, 256, 256)
mask_input = (mask_input * 2 - 1) * 10  # binary → ±10 logits

In [ ]:
with torch.inference_mode():
    masks, scores, _ = predictor.predict(
        mask_input=mask_input,
        multimask_output=False,
    )
best_mask = masks[0].astype(bool)


In [ ]:
def overlay_mask(xct, mask, color=(1, 0.2, 0.2), alpha=0.4):
    mask = mask.astype(bool)
    rgb = np.stack([xct] * 3, axis=-1)
    overlay = rgb.copy()
    overlay[mask] = (1 - alpha) * rgb[mask] + alpha * np.array(color)
    return np.clip(overlay, 0.0, 1.0)


fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(xct_img, cmap='gray');                 axs[0].set_title('XCT')
axs[1].imshow(mapA, cmap='gray');                    axs[1].set_title('XFuse mask (coarse)')
axs[2].imshow(overlay_mask(xct_img, best_mask));     axs[2].set_title(f'SAM2 refined (score={scores[0]:.3f})')
for ax in axs.flat: ax.set_axis_off()
plt.tight_layout()


In [ ]:
# Test CLIP

import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-H-14", pretrained="laion2b_s32b_b79k", device=DEVICE
)
tokenizer = open_clip.get_tokenizer("ViT-H-14")

def score_pca_map(thresholded_map, prompt, clip_model, clip_preprocess, device):
    pca_uint8 = (np.stack([thresholded_map] * 3, axis=-1).reshape(IMG_SIZE, IMG_SIZE, 3) * 255).astype(np.uint8)
    img = Image.fromarray(pca_uint8, mode='RGB')
    img_t = clip_preprocess(img).unsqueeze(0).to(device)
    text_t = tokenizer([prompt]).to(device)
    with torch.no_grad():
        img_f  = clip_model.encode_image(img_t).float()
        text_f = clip_model.encode_text(text_t).float()
    return (img_f @ text_f.T).item()

In [ ]:
PROMPT = "two circles"
PROMPT = "small blobs"

# threshold the PCA maps and score them with CLIP to see if they align with the prompt

# mapA = pcaed_lgA[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold
# mapB = pcaed_lgB[:, 0].reshape(IMG_SIZE, IMG_SIZE) > threshold

score_A = score_pca_map(mapA, PROMPT, clip_model, clip_preprocess, DEVICE)
score_B = score_pca_map(mapB, PROMPT, clip_model, clip_preprocess, DEVICE)
print(f"Phase A: {score_A:.3f}  |  Phase B: {score_B:.3f}")

In [ ]:
selected = mapA if score_A > score_B else mapB
print(f"→ Selected Phase {'A' if score_A > score_B else 'B'}")

In [ ]:
plt.imshow(selected.reshape(IMG_SIZE, IMG_SIZE, 1))
plt.title(f"Selected phase (A score={score_A:.3f}, B score={score_B:.3f})")
plt.axis('off')
plt.show()

In [ ]:
# --- SAM2 single pass ---
mask_input = cv2.resize(selected.astype(np.float32), (256, 256))[None]
mask_input = (mask_input * 2 - 1) * 10
with torch.inference_mode():
    masks, scores, _ = predictor.predict(mask_input=mask_input, multimask_output=False)
best_mask = masks[0].astype(bool)

fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(xct_img, cmap='gray');                 axs[0].set_title('XCT')
axs[1].imshow(mapA, cmap='gray');                    axs[1].set_title('XFuse mask (coarse)')
axs[2].imshow(overlay_mask(xct_img, best_mask));     axs[2].set_title(f'SAM2 refined (score={scores[0]:.3f})')
for ax in axs.flat: ax.set_axis_off()
plt.tight_layout()